In [1]:
import time
import os
import joblib
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from scipy.optimize import differential_evolution

In [5]:
# ----------------------------------------------------
# 0. 전역 설정 및 파일 로드 경로
# ----------------------------------------------------
N_ENSEMBLE = 10
TARGET_NAMES = ["Deflection(mm)", "Weight(kg)"]
MODEL_PATHS = [f'saved_models/pv_ensemble_model_{i+1}.keras' for i in range(N_ENSEMBLE)]
SCALER_DIR = '.' # 스케일러가 저장된 디렉토리
OUTPUT_DIR = 'de_optimization_results_pv'

# ----------------------------------------------------
# 1. 앙상블 모델 및 스케일러 로딩
# ----------------------------------------------------
try:
    ENSEMBLE_MODELS = [load_model(path, compile=False) for path in MODEL_PATHS]
    X_SCALER_LOADED = joblib.load(os.path.join(SCALER_DIR, 'scaler_X_pvmodule.joblib'))
    Y_SCALER_LOADED = joblib.load(os.path.join(SCALER_DIR, 'scaler_y_pvmodule.joblib'))
    print("앙상블 모델 및 스케일러 로딩 완료.")
except Exception as e:
    print(f"오류: 모델 또는 스케일러 로드 실패. 저장 경로 확인 필요. ({e})")
    raise

앙상블 모델 및 스케일러 로딩 완료.


C:\Users\admin\anaconda3\envs\py31010_auto\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
CONSTRAINTS = {
    "Deflection(mm)": {"min": None, "max": 12.3}, 
    "Weight(kg)": {"min": None, "max": 3.6}       
}

PENALTY_CONST = 100.0

In [9]:
# ----------------------------------------------------
# 2. 목적 함수 및 탐색 기록 수집 정의
# ----------------------------------------------------
DE_HISTORY = [] 

def objective_for_DE_constrained_wrapper(X):
    """ DE에 전달되며, 탐색 기록을 수집하는 Wrapper 함수 """
    a, b, c, d, e = X 
    
    X_input = np.array([[a, b, c, d, e]])
    X_scaled = X_SCALER_LOADED.transform(X_input)
    
    # 앙상블 평균 예측 (원본 물리적 값 반환)
    preds_raw = [model.predict(X_scaled, verbose=0)[0] for model in ENSEMBLE_MODELS]
    y_pred_mean_raw = np.mean(np.array(preds_raw), axis=0)

    # 공평한 F_obj 계산을 위한 스케일링
    y_pred_mean_scaled = Y_SCALER_LOADED.transform(y_pred_mean_raw.reshape(1, -1))[0]

    # 변위(음수)는 0에 가깝도록 "최대화(+1)", 무게(양수)는 "최소화(-1)"
    GOAL_SIGNS = np.array([1, -1]) 
    WEIGHTS = np.array([1.0, 1.0])
    F_obj_unpenalized = np.sum(GOAL_SIGNS * WEIGHTS * y_pred_mean_scaled)
    
    # 제약 조건 검사 및 벌칙 적용 (원본 물리적 값 기준)
    penalty = 0.0
    for i, target_name in enumerate(TARGET_NAMES):
        value = y_pred_mean_raw[i]
        
        # 처짐량은 절댓값으로 변환하여 검사
        if target_name == "Deflection(mm)":
            check_value = abs(value)
        else:
            check_value = value
                
        if target_name in CONSTRAINTS:
            max_req = CONSTRAINTS[target_name]["max"]
            if max_req is not None and check_value > max_req:
                penalty += PENALTY_CONST * (check_value - max_req)
    
    F_obj_constrained = F_obj_unpenalized - penalty
    
    # 탐색 기록 저장 (원본 값 저장)
    record = list(X) + list(y_pred_mean_raw) + [F_obj_constrained]
    DE_HISTORY.append(record)
    
    # scipy differential_evolution은 최소화를 수행하므로 부호 반전
    return -F_obj_constrained 

# ----------------------------------------------------
# 4. DE 최적화 실행 및 저장 (Main Logic)
# ----------------------------------------------------
if __name__ == '__main__':
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    start_time = time.time()
    
    # 입력 인자 물리적 탐색 공간
    INPUT_BOUNDS = [(1.5, 2.5), (8.0, 14.0), (1.5, 2.5), (1.7, 3.0), (2.5, 5.0)]
    MAX_ITER = 40  
    POP_SIZE = 10  
    
    print(f"\n--- 1단계: DE 최적화 시작 ---")
    print(f"--- 제약 조건 반영 최적 입력 인자 탐색 시작 (DE, Max Iter: {MAX_ITER}, Pop Size: {POP_SIZE}) ---\n")

    result_de = differential_evolution(
        func=objective_for_DE_constrained_wrapper, 
        bounds=INPUT_BOUNDS,
        strategy='best1bin',
        maxiter=MAX_ITER,
        popsize=POP_SIZE,
        seed=42
    )
    
    end_time = time.time()
    elapsed_time = end_time - start_time

    # 4. 탐색 기록 DataFrame 생성
    COLUMNS = ['a', 'b', 'c', 'd', 'e'] + TARGET_NAMES + ['F_obj']
    df_total = pd.DataFrame(DE_HISTORY, columns=COLUMNS).drop_duplicates().reset_index(drop=True)
    
    # 5. 최적점 계산
    max_F_obj = df_total['F_obj'].max()
    best_X_data = df_total[df_total['F_obj'] == max_F_obj].iloc[0]
    
    # 6. 파일 저장
    file_prefix = 'DE_Optimization_Result_PV'
    df_total.to_csv(os.path.join(OUTPUT_DIR, f'{file_prefix}_Total_Samples.csv'), index=False)
    pd.DataFrame(best_X_data).transpose().to_csv(os.path.join(OUTPUT_DIR, f'{file_prefix}_Best_Solution.csv'), index=False)

    # ----------------------------------------------------
    # 7. 최적 결과 콘솔 출력
    # ----------------------------------------------------
    y_pred_mean_optimal = best_X_data[TARGET_NAMES].values
    
    print("\n=== 제약 조건 만족 복합 목표 최대화 최적 입력 인자 조합 (DE) ===")
    print(f"    a: {best_X_data['a']:.4f}")
    print(f"    b: {best_X_data['b']:.4f}")
    print(f"    c: {best_X_data['c']:.4f}")
    print(f"    d: {best_X_data['d']:.4f}")
    print(f"    e: {best_X_data['e']:.4f}")
    print(f"최대 목적 함수 값 (F_obj): {max_F_obj:.4f}")
    print(f"경과 시간 : {elapsed_time:.2f} 초")

    print("\n--- 최적 해의 예측된 출력값 (Y) ---")
    for name, value in zip(TARGET_NAMES, y_pred_mean_optimal):
        
        direction = "⬇️ (Minimize Goal)" if name == "Weight(kg)" else "⬆️ (Maximize closer to 0)"
        
        # 제약 조건 만족 여부 표시 (절댓값 처리 재적용)
        check_value = abs(value) if name == "Deflection(mm)" else value
        constraint_info = ""
        
        if name in CONSTRAINTS:
             max_req = CONSTRAINTS[name]["max"]
             if max_req is not None and check_value > max_req:
                 constraint_info = "❌ Violated"
             else:
                 constraint_info = "✅ Satisfied"
                 
        print(f"    - **{name}:** {value:.4f} {direction} ({constraint_info})")
    
    print(f"Elapsed time : {elapsed_time:.2f}")
    print("\n✅ DE 최적화 완료. 데이터가 파일로 저장되었습니다.")


--- 1단계: DE 최적화 시작 ---
--- 제약 조건 반영 최적 입력 인자 탐색 시작 (DE, Max Iter: 40, Pop Size: 10) ---


=== 제약 조건 만족 복합 목표 최대화 최적 입력 인자 조합 (DE) ===
    a: 1.5025
    b: 13.9998
    c: 1.5035
    d: 2.9953
    e: 3.7516
최대 목적 함수 값 (F_obj): 0.6695
경과 시간 : 1495.29 초

--- 최적 해의 예측된 출력값 (Y) ---
    - **Deflection(mm):** -11.2469 ⬆️ (Maximize closer to 0) (✅ Satisfied)
    - **Weight(kg):** 3.5945 ⬇️ (Minimize Goal) (✅ Satisfied)
Elapsed time : 1495.29

✅ DE 최적화 완료. 데이터가 파일로 저장되었습니다.


In [11]:
# ----------------------------------------------------
# 8. 딥 앙상블의 불확실성 분석 (X_best 지점)
# ----------------------------------------------------
print("\n--- 딥 앙상블 불확실성 분석 시작 ---")

# 최적 입력 인자 배열 구성 (a, b, c, d, e)
X_optimal_de = np.array([[best_X_data['a'], best_X_data['b'], 
                          best_X_data['c'], best_X_data['d'], best_X_data['e']]])
X_optimal_scaled_de = X_SCALER_LOADED.transform(X_optimal_de)

# 앙상블 예측 (모델이 원본 물리적 값을 직접 반환함)
preds_raw_optimal_de = [model.predict(X_optimal_scaled_de, verbose=0)[0] for model in ENSEMBLE_MODELS]
preds_raw_arr_de = np.array(preds_raw_optimal_de)

# ★ 역변환(inverse_transform) 및 scale_ 곱하기 전면 삭제 ★
mean_original_de = np.mean(preds_raw_arr_de, axis=0)
std_original_de = np.std(preds_raw_arr_de, axis=0)

print("\n=== 📊 앙상블 예측 신뢰성 분석 (X_best 지점) ===")
print("  (95% 예측 구간: Mean ± 1.96 * Std)")

results_df_de = pd.DataFrame({
    'Output': TARGET_NAMES,
    'Mean (Pred)': mean_original_de,
    'Std (Uncertainty)': std_original_de,
    'Lower 95% PI': mean_original_de - 1.96 * std_original_de,
    'Upper 95% PI': mean_original_de + 1.96 * std_original_de
})

print(results_df_de.to_markdown(floatfmt=".4f"))

# ----------------------------------------------------
# 9. 국소적 검증 (Local Verification: ±1% Perturbation)
# ----------------------------------------------------
print("\n--- 국소적 검증(Local Verification)을 위한 주변 샘플링 시작 ---")

N_LOCAL_SAMPLES = 10 
TOLERANCE_FACTOR = 0.01 
local_f_obj_values_de = []

for i in range(N_LOCAL_SAMPLES):
    X_local_sample = []
    for key in ['a', 'b', 'c', 'd', 'e']:
        mean_val = best_X_data[key]
        # 각 변수별 ±1% 범위 내에서 무작위 추출
        low = mean_val * (1 - TOLERANCE_FACTOR)
        high = mean_val * (1 + TOLERANCE_FACTOR)
        X_local_sample.append(np.random.uniform(low, high))
    
    # DE의 목적 함수 wrapper 호출 (배열 형태로 전달)
    # wrapper 함수는 값을 최소화하기 위해 -F_obj를 반환하므로 다시 -를 붙여 원래 최대화 값으로 복원
    cost_val = objective_for_DE_constrained_wrapper(X_local_sample)
    local_f_obj_values_de.append(-cost_val)

print("\n=== 국소적 강건성 검증 (Local Verification ±1%) ===")
print(f"기준점 (X_best) F_obj: **{max_F_obj:.4f}**")
print(f"주변 샘플 ({N_LOCAL_SAMPLES}개) F_obj 범위: **{np.min(local_f_obj_values_de):.4f} ~ {np.max(local_f_obj_values_de):.4f}**")
print(f"주변 샘플 F_obj 평균: **{np.mean(local_f_obj_values_de):.4f}**")


--- 딥 앙상블 불확실성 분석 시작 ---

=== 📊 앙상블 예측 신뢰성 분석 (X_best 지점) ===
  (95% 예측 구간: Mean ± 1.96 * Std)
|    | Output         |   Mean (Pred) |   Std (Uncertainty) |   Lower 95% PI |   Upper 95% PI |
|---:|:---------------|--------------:|--------------------:|---------------:|---------------:|
|  0 | Deflection(mm) |      -11.2469 |              0.0731 |       -11.3901 |       -11.1037 |
|  1 | Weight(kg)     |        3.5945 |              0.0407 |         3.5146 |         3.6743 |

--- 국소적 검증(Local Verification)을 위한 주변 샘플링 시작 ---

=== 국소적 강건성 검증 (Local Verification ±1%) ===
기준점 (X_best) F_obj: **0.6695**
주변 샘플 (10개) F_obj 범위: **0.5175 ~ 0.6821**
주변 샘플 F_obj 평균: **0.6381**
